# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maahilkureshi-design/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words.** Rank pages by how far their last 30 days fell compared with their previous 30 days, after subtracting what the page's whole client did over the same windows. The page with the biggest relative fall is first. The rule uses only the past (nothing after the snapshot date), needs no model, and anyone can check it by hand.

**Reason codes it can output** (a page can get up to three):

- `FALLING_VS_CLIENT`: last 30 days at least a third lower than the previous 30, relative to its client.
- `POSITION_SLIPPING`: average position got worse by 3 places or more.
- `LARGE_IMPRESSION_LOSS`: lost 500 or more impressions compared with the previous 30 days.
- `LOW_VISIBILITY`: under 200 impressions in the last 30 days, so counts are small and noisy.

I score the queue on the validation snapshot (2026-04-30), where the outcome of the next 30 days is known, so the review can say which picks held up. The rule is not tuned on any outcome.

In [1]:
import os, json, shutil
import numpy as np
import pandas as pd
import duckdb
from sklearn.metrics import average_precision_score

# The saved snapshot files and shared outputs live in Google Drive (MyDrive/flyrank_capstone),
# so this notebook can run in its own session after capstone.ipynb has finished.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT, ON_COLAB = '/content/drive/MyDrive/flyrank_capstone', True
except ImportError:
    DRIVE_ROOT, ON_COLAB = '.', False
CACHE_DIR, EVAL_DIR, OUT_DIR = 'snap_cache', 'eval_cache', f'{DRIVE_ROOT}/outputs'
if ON_COLAB:
    assert os.path.exists(f'{DRIVE_ROOT}/snap_cache'), 'Not found in Google Drive: run capstone.ipynb first, then its last cell (copy to Drive).'
    shutil.copytree(f'{DRIVE_ROOT}/snap_cache', CACHE_DIR, dirs_exist_ok=True)      # local copy: faster than reading from Drive
    if os.path.exists(f'{DRIVE_ROOT}/eval_cache'):
        shutil.copytree(f'{DRIVE_ROOT}/eval_cache', EVAL_DIR, dirs_exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
n_files = len([f for f in os.listdir(CACHE_DIR) if f.startswith('snap_') and f.endswith('.parquet')]) if os.path.exists(CACHE_DIR) else 0
assert n_files == 12, f'{n_files} snapshot files found, expected 12: run capstone.ipynb and its last cell (copy to Drive) first.'

LABEL_DAYS = 30
K_LIST = [20, 50]
MIN_CLIENT_PAGES_MODEL = 200
SPIKE_RULE = "imp_last30 >= 2 * GREATEST(imp_prev30, imp_prev60)"
UNIVERSE = f"n_elig >= {MIN_CLIENT_PAGES_MODEL} AND NOT ({SPIKE_RULE})"
PAST_COLS = ['imp_90', 'clicks_90', 'ctr_90', 'ctr_last30', 'pos_90', 'pos_last30', 'pos_prev30', 'pos_change',
             'imp_last30', 'imp_prev30', 'imp_prev60', 'active_days_90', 'cv_90',
             'log_mom', 'log_mom_rel', 'log_mom_prev', 'log_mom_prev_rel', 'page_age_days']
STATIC_COLS = ['word_count', 'char_count', 'content_type', 'main_intent', 'competition_level', 'competition',
               'search_volume', 'cpc', 'backlinks', 'category_count', 'keyword_token_count', 'url_char_count']
mcon = duckdb.connect()

def _clean(d):
    d['snap_t'] = pd.to_datetime(d['snap_t'])
    for c in PAST_COLS:
        if pd.api.types.is_timedelta64_dtype(d[c]):
            d[c] = d[c].dt.days
    return d

def load_train():
    cols = ', '.join(['snap_t', 'client_hash_id', 'y'] + PAST_COLS)
    return _clean(mcon.sql(f"SELECT {cols} FROM read_parquet('{CACHE_DIR}/snap_*.parquet') WHERE {UNIVERSE}").df())

def load_eval(name):
    path = f'{EVAL_DIR}/eval_{name}.parquet'
    assert os.path.exists(path), f'{path} is missing: run the build cells of capstone.ipynb first'
    cols = ', '.join(['snap_t', 'client_hash_id', 'y'] + PAST_COLS)
    return _clean(mcon.sql(f"SELECT {cols} FROM read_parquet('{path}') WHERE {UNIVERSE}").df())

def eval_rule(df, score_col, ks=K_LIST):
    rows = []
    for t, g in df.groupby('snap_t'):
        g = g.sort_values(score_col, ascending=False, kind='mergesort')
        base = g['y'].mean()
        row = {'snap_t': str(t)[:10], 'n': len(g), 'base_rate': round(base, 3)}
        for k in ks:
            p = g['y'].head(k).mean()
            row[f'p@{k}'] = round(p, 3)
            row[f'lift@{k}'] = round(p / base, 2) if base > 0 else np.nan
        ap = average_precision_score(g['y'], g[score_col]) if g['y'].sum() > 0 else np.nan
        row['ap'] = round(ap, 3)
        row['ap_lift'] = round(ap / base, 2) if base > 0 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

RULE = "Rank pages by relative momentum: the most negative log_mom_rel (fell most versus the client) comes first."

REASON_RULES = {
    'FALLING_VS_CLIENT': lambda d: d['log_mom_rel'] <= -0.4,
    'POSITION_SLIPPING': lambda d: d['pos_change'] >= 3,
    'LARGE_IMPRESSION_LOSS': lambda d: (d['imp_prev30'] - d['imp_last30']) >= 500,
    'LOW_VISIBILITY': lambda d: d['imp_last30'] < 200,
}
WRONG_IF = {
    'FALLING_VS_CLIENT': 'the fall is a return to normal after an earlier spike, or a client-wide shift the benchmark missed',
    'POSITION_SLIPPING': 'position moved on a few noisy days at low impressions',
    'LARGE_IMPRESSION_LOSS': 'the previous month was a one-off peak',
    'LOW_VISIBILITY': 'the counts are so small that normal noise explains the change',
}
print(RULE)
print('Reason codes:', list(REASON_RULES))

Mounted at /content/drive
Rank pages by relative momentum: the most negative log_mom_rel (fell most versus the client) comes first.
Reason codes: ['FALLING_VS_CLIENT', 'POSITION_SLIPPING', 'LARGE_IMPRESSION_LOSS', 'LOW_VISIBILITY']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The code scores every page in the validation snapshot, ranks them, attaches reason codes and a suggested review action, and writes the queue to `work/outputs/baseline_action_score.csv`. The file has no page or client identifiers, only a client size rank and the numbers behind each pick.*

In [2]:
val = load_eval('validation')
val['score'] = -val['log_mom_rel']

conds = pd.DataFrame({k: f(val).fillna(False) for k, f in REASON_RULES.items()})
val['reason_codes'] = conds.apply(lambda r: ', '.join([k for k, v in r.items() if v][:3]), axis=1)

val['action'] = 'monitor'
val.loc[conds['FALLING_VS_CLIENT'], 'action'] = 'review: improve or monitor'
val.loc[conds['FALLING_VS_CLIENT'] & (val['imp_last30'] >= 1000), 'action'] = 'review now: improve or refresh'
val.loc[conds['FALLING_VS_CLIENT'] & conds['LOW_VISIBILITY'], 'action'] = 'review for merge or prune'

client_rank = val.groupby('client_hash_id').size().rank(ascending=False, method='first').astype(int)
val['client_rank'] = val['client_hash_id'].map(client_rank)

queue = val.sort_values('score', ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1
out_cols = ['rank', 'client_rank', 'score', 'imp_last30', 'imp_prev30', 'log_mom_rel', 'reason_codes', 'action']
queue[out_cols].round(4).to_csv(os.path.join(OUT_DIR, 'baseline_action_score.csv'), index=False)
print(f"{len(queue):,} pages ranked; written to {OUT_DIR}/baseline_action_score.csv")
print(queue['action'].value_counts().to_string())

# How does the rule do on the known outcome? (a random queue has lift 1.0)
print()
print(eval_rule(val.assign(score_rule=val['score']), 'score_rule').to_string(index=False))

60,463 pages ranked; written to /content/drive/MyDrive/flyrank_capstone/outputs/baseline_action_score.csv
action
monitor                           42195
review: improve or monitor         8517
review for merge or prune          4886
review now: improve or refresh     4865

    snap_t     n  base_rate  p@20  lift@20  p@50  lift@50    ap  ap_lift
2026-04-30 60463      0.169   0.2     1.19  0.14     0.83 0.188     1.11


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

For each of the top 20 picks: the suggested action, the reason code, a confidence note based on volume, what would make the pick wrong, and whether the page actually fell to half or less of its client's trend in the next 30 days (1 = yes). The outcome column is there to judge the rule, not something the rule could see.

In [3]:
def confidence(n):
    if n < 200:
        return 'low: under 200 impressions'
    if n < 1000:
        return 'medium'
    return 'higher: 1,000+ impressions'

review = queue.head(20).copy()
review['confidence'] = review['imp_last30'].map(confidence)
review['what_would_make_it_wrong'] = review['reason_codes'].map(
    lambda s: WRONG_IF.get(s.split(', ')[0], 'no reason code fired, so this is ordering only') if s else 'no reason code fired, so this is ordering only')
review['declined_next_30d'] = review['y']
cols = ['rank', 'client_rank', 'imp_last30', 'imp_prev30', 'reason_codes', 'action', 'confidence', 'what_would_make_it_wrong', 'declined_next_30d']
print(review[cols].round(2).to_string(index=False))
print(f"\nOf the top 20, {int(review['declined_next_30d'].sum())} declined in the next 30 days (base rate {val['y'].mean():.3f}).")

 rank  client_rank  imp_last30  imp_prev30                                                reason_codes                         action                 confidence                                                                           what_would_make_it_wrong  declined_next_30d
    1            1       188.0     54887.0 FALLING_VS_CLIENT, POSITION_SLIPPING, LARGE_IMPRESSION_LOSS      review for merge or prune low: under 200 impressions the fall is a return to normal after an earlier spike, or a client-wide shift the benchmark missed                  0
    2            3       124.0     12358.0    FALLING_VS_CLIENT, LARGE_IMPRESSION_LOSS, LOW_VISIBILITY      review for merge or prune low: under 200 impressions the fall is a return to normal after an earlier spike, or a client-wide shift the benchmark missed                  0
    3            2       124.0     12996.0    FALLING_VS_CLIENT, LARGE_IMPRESSION_LOSS, LOW_VISIBILITY      review for merge or prune low: under 200 impressions th

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks.** The picks that did not decline are listed below with their reason codes, so the failure pattern is visible. Rules that rely on one past movement tend to fail when the movement was a return to normal, or when counts are small.

**Leakage check.** The score is computed from `log_mom_rel`, a past-only feature. No product flag, no label column and no window after the snapshot date is used. The assertions below confirm it.

In [4]:
weak = review[review['declined_next_30d'] == 0]
print(f"{len(weak)} weak picks in the top 20:")
print(weak[['rank', 'client_rank', 'imp_last30', 'imp_prev30', 'reason_codes', 'confidence']].round(2).to_string(index=False))

PRODUCT_FLAGS = ['health_score', 'priority_score', 'action_type', 'refresh_priority', 'trend_direction', 'trend_pct']
assert 'log_mom_rel' in PAST_COLS, 'the score must come from a past-only feature'
assert not set(PAST_COLS) & set(PRODUCT_FLAGS + ['y', 'rel', 'rel_h1', 'rel_h2']), 'a product flag or label column is in the features'
assert (queue['score'] == -queue['log_mom_rel']).all(), 'the score uses something other than log_mom_rel'
print('\nLeakage check passed: the score uses only log_mom_rel, a feature built from data up to the snapshot date.')

16 weak picks in the top 20:
 rank  client_rank  imp_last30  imp_prev30                                                reason_codes                 confidence
    1            1       188.0     54887.0 FALLING_VS_CLIENT, POSITION_SLIPPING, LARGE_IMPRESSION_LOSS low: under 200 impressions
    2            3       124.0     12358.0    FALLING_VS_CLIENT, LARGE_IMPRESSION_LOSS, LOW_VISIBILITY low: under 200 impressions
    4            3       170.0     12455.0 FALLING_VS_CLIENT, POSITION_SLIPPING, LARGE_IMPRESSION_LOSS low: under 200 impressions
    7            2       120.0      8283.0 FALLING_VS_CLIENT, POSITION_SLIPPING, LARGE_IMPRESSION_LOSS low: under 200 impressions
    8            6       118.0      2503.0 FALLING_VS_CLIENT, POSITION_SLIPPING, LARGE_IMPRESSION_LOSS low: under 200 impressions
   10            3       134.0      6617.0 FALLING_VS_CLIENT, POSITION_SLIPPING, LARGE_IMPRESSION_LOSS low: under 200 impressions
   11            2       115.0      6170.0 FALLING_VS_CLIENT,

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.